**Netflix --> Movie Recommendation System**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as mlt
import seaborn as sns

In [ ]:
print(np.__version__)

1.26.4


**Downgrade Numpy --> surprise is not compatable with current version**

In [ ]:
pip install scikit-surprise

In [ ]:
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

In [ ]:
#Downgrade Numpy --> surprise is not compatable with current version
!pip install numpy==1.26.4

In [ ]:
print(np.__version__)

1.26.4


**Importing Library**

In [ ]:
nd=pd.read_csv("/content/combined_data_1.txt",
                            header=None,
                            names=["Cust_ID","Rating"],
                            usecols=[0,1])  # [0,1]-> use 1st and 2nd col,header = none->  The file dosent have col names

In [ ]:
nd

,Cust_ID,Rating
0,1:,NaN
1,1488844,3.0
2,822109,5.0
3,885013,4.0
4,30878,4.0
...,...,...
17428452,1516711,2.0
17428453,2106620,3.0
17428454,227945,4.0
17428455,1606088,5.0


In [ ]:
nd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17428457 entries, 0 to 17428456
Data columns (total 2 columns):
 #   Column   Dtype  
---  ------   -----  
 0   Cust_ID  object 
 1   Rating   float64
dtypes: float64(1), object(1)
memory usage: 265.9+ MB


In [ ]:
nd=pd.read_csv("/content/combined_data_1.txt",
                            header=None,
                            names=["Cust_ID","Rating"],
                            usecols=[0,1])
nd.isnull().sum()

,0
Cust_ID,0
Rating,3348


In [ ]:
movie_id=[]
movie_count=0

for i in nd["Cust_ID"]:
  if ":" in i:
    movie_count+=1
    # Replace the : with "" and then convert to int( "1:" -> 1)
    id=int(i.replace(":",""))
  movie_id.append(id)


print("total number of movies=",movie_count)

total number of movies= 3347


In [ ]:
print(movie_id[0],movie_id[-1])

1 3347


In [ ]:
nd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17428457 entries, 0 to 17428456
Data columns (total 2 columns):
 #   Column   Dtype  
---  ------   -----  
 0   Cust_ID  object 
 1   Rating   float64
dtypes: float64(1), object(1)
memory usage: 265.9+ MB


In [ ]:
# Create a new Column-> MovieID

nd["MovieID"]=movie_id
nd

,Cust_ID,Rating,MovieID
0,1:,NaN,1
1,1488844,3.0,1
2,822109,5.0,1
3,885013,4.0,1
4,30878,4.0,1
...,...,...,...
17428452,1516711,2.0,3347
17428453,2106620,3.0,3347
17428454,227945,4.0,3347
17428455,1606088,5.0,3347


In [ ]:
nd.dropna(inplace=True)
nd

,Cust_ID,Rating,MovieID
1,1488844,3.0,1
2,822109,5.0,1
3,885013,4.0,1
4,30878,4.0,1
5,823519,3.0,1
...,...,...,...
17428451,2402249,2.0,3347
17428452,1516711,2.0,3347
17428453,2106620,3.0,3347
17428454,227945,4.0,3347


In [ ]:
nd.dtypes

,0
Cust_ID,object
Rating,float64
MovieID,int64


In [ ]:
#Change data type od customer id

nd["Cust_ID"]=nd["Cust_ID"].astype(int)
nd.dtypes

,0
Cust_ID,int64
Rating,float64
MovieID,int64


In [ ]:
# 1) Remove the users that have rated less number of movies

# Number of ratings done by each customer
data_customer= nd["Cust_ID"].value_counts()
data_customer

,count
Cust_ID,
305344,3318
387418,3291
2439493,3125
1664010,2982
2118461,2810
...,...
917536,1
2000123,1
2159770,1


In [ ]:
# Benchmark for customer rating
# - min number of ratings that a customer should have
# We want top 40 % customers( in terms of number of ratings)

cust_benchmark=round(data_customer.quantile(0.6))

# This will give a number below which 60% of the people lie

cust_benchmark

# A customer should have a min of 36 ratings

26

In [ ]:
# 2) Removing the movies with less number of ratings

data_movie=nd["MovieID"].value_counts()
data_movie

,count
MovieID,
1905,193941
2152,162597
571,154832
2452,149866
1962,145519
...,...
1416,55
1858,54
2805,46


In [ ]:
# We want top 40 % movies( in terms of number of ratings)

movie_benchmark=round(data_movie.quantile(0.6))

# This will give a number below which 60% of the people lie

movie_benchmark

# Movie should atleast have 911 ratings

894

In [ ]:
# List of customers to drop from the dataset

drop_cust= data_customer[ data_customer<cust_benchmark ].index
drop_cust

Index([ 228048,  974974, 1726143, 1790244, 1272985, 2305789, 2202223, 1317719,
       1882591,  262604,
       ...
        413019,  211699, 2544350,  244702,  249800,  917536, 2000123, 2159770,
       1199332, 2177298],
      dtype='int64', name='Cust_ID', length=276080)

In [ ]:
#Movies to drop
drop_movie= data_movie[ data_movie<movie_benchmark ].index
drop_movie

Index([2561, 2726, 3176,  212, 2673, 1932,  694, 2628, 1413, 1128,
       ...
       1591, 1445, 2744, 2291, 1383, 1416, 1858, 2805,  820,  915],
      dtype='int64', name='MovieID', length=2007)

In [ ]:
#Filter out
nd = nd[~nd["Cust_ID"].isin (drop_cust)]
nd

,Cust_ID,Rating,MovieID
1,1488844,3.0,1
3,885013,4.0,1
4,30878,4.0,1
5,823519,3.0,1
6,893988,3.0,1
...,...,...,...
17428451,2402249,2.0,3347
17428452,1516711,2.0,3347
17428453,2106620,3.0,3347
17428454,227945,4.0,3347


In [ ]:
nd = nd[~nd["Cust_ID"].isin (drop_movie)]
nd

,Cust_ID,Rating,MovieID
1,1488844,3.0,1
3,885013,4.0,1
4,30878,4.0,1
5,823519,3.0,1
6,893988,3.0,1
...,...,...,...
17428451,2402249,2.0,3347
17428452,1516711,2.0,3347
17428453,2106620,3.0,3347
17428454,227945,4.0,3347


###Load the movie names dataset

In [ ]:
df_names= pd.read_csv("/content/movies.csv", usecols=[0,1,2])
df_names

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
27273,131254,Kein Bund für's Leben (2007),Comedy
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy
27275,131258,The Pirates (2014),Adventure
27276,131260,Rentun Ruusu (2001),(no genres listed)


In [ ]:
# Import the libraries
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

In [ ]:
reader=Reader(rating_scale=(1,5)) # range of rating

#Dataset

In [ ]:
data= Dataset.load_from_df(nd[["Cust_ID","MovieID","Rating"]][:100000],reader)
data

In [ ]:
model=SVD()

**Cross Validation**

It's a model evaluation step( how good the model is performing).

Model uses RMSE score( The lesser the better) .

SPlit the data into train and test
Train the model or Train data
Test the model on test data
Calculate th RMSE
This is to check how good the model is performing.

In [ ]:
cross_validate(model,data,measures=["RMSE"],cv=3)

{'test_rmse': array([1.01541474, 1.02690864, 1.02428652]),
 'fit_time': (1.7616417407989502, 1.6822054386138916, 1.175593614578247),
 'test_time': (0.32607245445251465, 0.39605236053466797, 0.167219877243042)}

#Train the Model

In [ ]:
traindata= data.build_full_trainset()

# build_full_trainset() -> turns the ratings into format that Model understands

# Train model
model.fit(traindata)

In [ ]:
#Filter the data for finding a specific user to whom we are going to suggest movies

user_rating=nd[nd['Cust_ID']==1331154]
user_rating

,Cust_ID,Rating,MovieID
697,1331154,4.0,3
5178,1331154,4.0,8
31460,1331154,3.0,18
92840,1331154,4.0,30
224761,1331154,3.0,44
...,...,...,...
17174834,1331154,3.0,3315
17210710,1331154,4.0,3320
17293992,1331154,3.0,3326
17319044,1331154,5.0,3333


#User Base Recommendation

In [ ]:
user_1331154=df_names.copy()
user_1331154

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
27273,131254,Kein Bund für's Leben (2007),Comedy
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy
27275,131258,The Pirates (2014),Adventure
27276,131260,Rentun Ruusu (2001),(no genres listed)


In [ ]:
#Remove the less rated movies from 2nd dataset also

user_1331154=user_1331154[~user_1331154['movieId'].isin(drop_movie)]

In [ ]:
#Prediction part

user_1331154['Estimated_Score']=user_1331154['movieId'].apply(lambda x:model.predict(1331154,x).est)
#Estimate out of 5
#Using movieId column, the trained SVD model predicts the rating that user 1331154 would give to movie "x",
#.est- gives an estimated value between 1-5

In [ ]:
#Display top 5 movies with the highest estimate score that user 1331154 may like?

user_1331154.sort_values("Estimated_Score",ascending=False,inplace=True)

In [ ]:
user_1331154.head(5)

,movieId,title,genres,Estimated_Score
12,13,Balto (1995),Adventure|Animation|Children,4.616772
27,28,Persuasion (1995),Drama|Romance,3.983060
24,25,Leaving Las Vegas (1995),Drama|Romance,3.900242
29,30,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Crime|Drama,3.841294
2,3,Grumpier Old Men (1995),Comedy|Romance,3.813736


### Generate Recommendations for a Different User

In [ ]:
# Enter the User ID for whom you want recommendations
new_user_id = 1488844 # You can change this to any valid user ID from your dataset

In [ ]:
# Movies already rated by the new user
rated_movies_new_user = nd[nd['Cust_ID'] == new_user_id]['MovieID'].unique()

# Movies not yet rated by the new user
unrated_movies_new_user = df_names[~df_names['movieId'].isin(rated_movies_new_user)].copy()

# Predict ratings for all unrated movies for the new user
unrated_movies_new_user['Estimated_Score'] = unrated_movies_new_user['movieId'].apply(
    lambda movie_id: model.predict(new_user_id, movie_id).est
)

# Top 10 recommendations for the new user
recommendations_new_user = (
    unrated_movies_new_user
    .sort_values('Estimated_Score', ascending=False)
    .head(10)
)

print(f"Top 10 recommendations for User ID: {new_user_id}")
print(recommendations_new_user[['movieId', 'title', 'Estimated_Score']])

Top 10 recommendations for User ID: 1488844
       movieId                                              title  \
12          13                                       Balto (1995)   
24          25                           Leaving Las Vegas (1995)   
4            5                 Father of the Bride Part II (1995)   
27          28                                  Persuasion (1995)   
5            6                                        Heat (1995)   
2            3                            Grumpier Old Men (1995)   
22          23                                   Assassins (1995)   
17          18                                  Four Rooms (1995)   
14          15                            Cutthroat Island (1995)   
18316    91537  Don't Worry, I'm Fine (Je vais bien, ne t'en f...   

       Estimated_Score  
12            4.255463  
24            4.253139  
4             3.934854  
27            3.733533  
5             3.613517  
2             3.610617  
22            3.56813